# 恒星分类 —— Kaggle Playground Series S6E6

[**斯隆数字化巡天（SDSS）**](https://en.wikipedia.org/wiki/Sloan_Digital_Sky_Survey) 用五个波段拍摄了整片天空，并测量了数百万个天体的光谱。本 notebook 使用该星表在 Kaggle *Playground Series* 上的版本。

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/8/82/SDSS_Camera.jpg" width="300">
<br>
<em>(The SDSS Imaging Camera, the heart of the 2.5m Telescope. Source: Wikimedia Commons)</em>
</center>

## 任务

给定少量测光数据，判断我们看到的是哪一种天体：

| 列名 | 角色 | 说明 |
| :--- | :--- | :--- |
| `u, g, r, i, z` | 特征 | 五个颜色滤光片下的亮度（星等） |
| `redshift` | 特征 | 天体远离我们的速度；恒星的 redshift ≈ 0 |
| `alpha, delta` | 特征 | 天球坐标 |
| `spectral_type`, `galaxy_population` | 特征 | 类别型元数据 |
| `class` | **目标** | **STAR**、**GALAXY** 或 **QSO**（类星体） |

训练集 577,347 行，测试集 247,435 行。

## 指标：平衡准确率（不是普通准确率！）

这是整个 notebook 中最重要的一件事。Kaggle 用下面的公式给本比赛打分：

$$\text{Balanced Accuracy} = \frac{1}{C}\sum_{c=1}^{C}\frac{TP_c}{TP_c+FN_c} = \frac{1}{C}\sum_{c=1}^{C}\text{Recall}_c$$

用文字说就是：**分别计算每个类别的召回率，再取平均。** 无论一个类别多么稀有，它都占同样的权重。

为什么这很重要？看看标签分布：

| 类别 | 数量 | 占比 |
| :--- | ---: | ---: |
| GALAXY | 377,480 | 65.4 % |
| QSO | 117,143 | 20.3 % |
| STAR | 82,724 | **14.3 %** |

一个偷懒的模型只要对所有样本都回答 “GALAXY”，就能拿到 **65% 的准确率**，但平衡准确率只有 **33%**。普通准确率在奖励“无视稀有的 STAR”，而平衡准确率拒绝这样做。下面的一切都围绕它来设计：我们对损失函数加权、用平衡准确率来*挑选* checkpoint，并同时报告 `accuracy`、`balanced_acc`、`macro_f1` 三个数字。

## 本 notebook 的组织方式

我们先**从零手写**经典算法（不借助库的魔法）来建立直觉，然后切换到完整的比赛流水线，使用深度模型。

| 小节 | 主题 | 一句话思想 |
| :--- | :--- | :--- |
| **1** | 数据预处理与可视化 | 标签编码、min–max 归一化、观察 redshift 分布 |
| **2** | 决策树桩 | 一条规则：用 redshift 阈值区分 STAR 与非 STAR |
| **3** | K 近邻 | 多分类的邻居投票 |
| **4** | K-Means | 无监督结构 + 肘部法 |
| **5** | 逐类别指标 | 每个类别的 precision / recall / F1 |
| — | 经典基线 | 把以上全部放到真实比赛数据上跑一遍 |
| **6** | 比赛环境 | 指标、无泄漏划分、特征工程、共享训练器 |
| **7** | 梯度提升树 | 表格数据上最强的经典模型 |
| **8** | MLP | 用反向传播训练的多层神经元 |
| **9** | RNN | 把特征当成*序列*，携带一个隐状态 |
| **10** | LSTM | 用门控记忆解决 RNN 的梯度消失 |
| **11** | Transformer | 在特征“词元”上做自注意力 |
| **12** | GAN | 生成少数类合成样本，对抗类别不均衡 |
| **13** | 集成 | 融合所有模型，产出最终 `submission.csv` |
| **14** | 收尾 | 结果、经验教训、扩展方向 |

> **在 Colab 上运行。** 把 `train.csv` 与 `test.csv` 放在 notebook 同级目录（或挂载 Drive），然后选择 **运行时 → 更改运行时类型 → GPU (T4)**。notebook 会自动找到这些文件，并告诉你它正在使用哪个设备。

> **两套特征。** 第 1–5 节使用一个小特征集（8 列），让手写算法的数学保持易读。从第 6 节起，我们为真正参与角逐的模型使用更丰富的工程化特征集（14 列）。


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Dict, Tuple, List

# --- Locate the competition CSVs (works on Colab and locally) ----------
CANDIDATE_DIRS = [".", "playground-series-s6e6", "/content",
                  "/content/playground-series-s6e6", "/content/drive/MyDrive"]

def find_csv(name):
    for d in CANDIDATE_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    hits = glob.glob(os.path.join("**", name), recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Could not find {name}. Put {name} next to the notebook.")


## 第 1 节 —— 数据预处理与可视化

`class` 列是字符串，而模型只吃数字，所以我们做标签编码：`STAR → 0`、`GALAXY → 1`、`QSO → 2`。各特征的量纲差异很大，因此我们再做 **min–max 归一化**，把它们压到 $[0, 1]$：

$$x_{norm} = \frac{x - \min(x)}{\max(x) - \min(x)}$$

有两个细节必须留意：

* **数据泄漏。** min/max 只能在训练集上计算，再套用到测试集。若在全部数据上一起算，就会把测试集的信息悄悄泄漏进模型。这正是 `preprocess_data` 要接收 `stats` 字典的原因：第一次调用（训练集）负责计算并返回它，之后的调用（测试集）复用它。
* **除零问题。** 如果某个特征在训练集上是常数，则 `max - min = 0`，会触发除零；此时该列直接置 0。

我们还画出每个类别的 `redshift` 分布。这一张图就解释了第 2 节的决策树桩为什么有效：恒星挤在 redshift ≈ 0 附近，而星系与类星体散布在远大于 0 的区间。


In [ ]:
def preprocess_data(df: pd.DataFrame, stats: Optional[Dict[str, Dict[str, float]]] = None) -> Tuple[np.ndarray, np.ndarray, Dict[str, Dict[str, float]]]:
    """
    Preprocesses the dataset by encoding labels and normalizing features.

    Args:
        df: Input DataFrame.
        stats: Dictionary containing min/max for normalization (optional).

    Returns:
        X: Processed features as a numpy array of shape (N, D).
        y: Labels as a numpy array of shape (N,).
        stats: Dictionary containing the train-set min/max used for normalization.
    """
    df = df.copy()
    # --- Label Encoding ---
    class_map = {'STAR': 0, 'GALAXY': 1, 'QSO': 2}
    spectral_map = {'M': 0, 'A/F': 0.25, 'O/B': 0.5,"G/K":0.75}
    population_map={'Red_Sequence':0,"Blue_Cloud":2}

    # 1. Apply class mapping to 'class' column
    if df.shape[1]==12:
      df["class"]=df["class"].map(class_map)
    df["spectral_type"]=df["spectral_type"].map(spectral_map)
    df["galaxy_population"]=df["galaxy_population"].map(population_map)
    # --- Min-Max Normalization ---
    feature_cols = ['alpha','delta','u', 'g', 'r', 'i', 'z', 'redshift']
    # 2. Compute or use existing stats for normalization
    #    If stats is None, compute min/max for each column in feature_cols and store in stats.
    #    Structure of stats: {'col_name': {'min': val, 'max': val}, ...}
    if stats is None:
      stats={}
      for col in feature_cols:
        stats[col]={'min':df[col].min(),'max':df[col].max()}
    # 3. Apply normalization to feature_cols
    for col in feature_cols:
      if stats[col]['max']-stats[col]['min']!=0:
        df[col]=(df[col]-stats[col]['min'])/(stats[col]['max']-stats[col]['min'])
      else:
        df[col]=0
    df["redshift"]*=3
    # 4. Extract X (features) and y (labels)
    # X should be a numpy array of shape (N, D)
    # y should be a numpy array of shape (N,)

    X = np.array(df[feature_cols].values)
    if df.shape[1]==12:
      y=np.array(df['class'])
    else:
      y=np.array([])

    return X, y, stats

def plot_redshift_distribution(df: pd.DataFrame) -> None:
    """
    Plots the distribution (histogram) of redshift for each class.

    Args:
        df: The original DataFrame containing 'redshift' and 'class'.
    """
    # 1. Iterate through the 3 classes (STAR, GALAXY, QSO)
    # Define common bins for comparison
    d_star=df[df['class']=="STAR"]
    d_galaxy=df[df['class']=="GALAXY"]
    d_qso=df[df['class']=="QSO"]
    bins=np.linspace(df['redshift'].min(),df['redshift'].max())
    # 2. Plot a histogram of redshift for that class (use alpha=0.5 for transparency)
    # Use log=True to visualize imbalanced classes (QSO vs Star)
    plt.hist(d_star['redshift'],bins=bins,alpha=0.5,label='STAR',log=True)
    plt.hist(d_galaxy['redshift'],bins=bins,alpha=0.5,label='GALAXY',log=True)
    plt.hist(d_qso['redshift'],bins=bins,alpha=0.5,label='QSO',log=True)
    # 3. Add legend and labels
    plt.title('Redshift Distribution by Class (Log Scale)')
    plt.xlabel('Redshift')
    plt.ylabel('Density (Log Scale)')
    plt.legend()


## 第 2 节 —— 决策树桩（STAR vs 非 STAR）

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/e4/Redshift_blueshift.svg/800px-Redshift_blueshift.svg.png" width="600">
<br>
<em>(Redshift and Blueshift. Source: Wikimedia Commons)</em>
</center>

决策树桩是最简单的模型：一个特征、一个阈值、一条规则。

### 背后的物理

恒星属于我们自己的星系，相对于我们几乎不动，redshift ≈ 0。远处的星系与类星体快速退行（哈勃定律），redshift 明显更大。因此，仅凭 `redshift` 的一个阈值就能把 STAR 和其余天体分开：

$$\hat{y} = \begin{cases} \text{STAR }(0) & \text{redshift} < T \\ \text{非 STAR }(1) & \text{redshift} \ge T \end{cases}$$

*这一步我们把 GALAXY $(1)$ 与 QSO $(2)$ 合并成一个“非 STAR” $(1)$ 类。*

### 找最优阈值

把样本按 redshift 排序，从左到右滑动分割点，保留准确率最高的那个阈值。下面的实现用一次向量化完成：借助 `np.cumsum` 前缀和，可以一次性算出**所有**候选阈值的得分，而不必对每个阈值重新算一遍准确率。只有 redshift 取值真正发生变化的位置才作为候选阈值。


In [ ]:
import numpy as np
class DecisionStump:
    def __init__(self):
        self.threshold = None
        self.feature_idx = -1  # 'redshift' is always the last column of X

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """
        Finds the optimal threshold T for the redshift feature to classify Star (0) vs Non-Star (>0).
        """
        # 1. 二值化标签：0 → 0 (Star), 1/2 → 1 (Non-Star)
        y_binary = (y != 0).astype(int)
        # 2. 提取 redshift 列
        x = X[:, self.feature_idx]

        # 3. 按特征值排序
        sort_idx = np.argsort(x)
        x_sorted = x[sort_idx]
        y_sorted = y_binary[sort_idx]

        N = len(y_sorted)
        total_ones = np.sum(y_sorted)

        # 4. 前缀和（方便快速计算左边/右边 1 的个数）
        cumsum = np.cumsum(y_sorted)   # cumsum[i] = sum(y_sorted[0..i])

        # 5. 计算每个分割位置（左子包含 0..i-1，右子包含 i..N-1）的正确分类数
        # 正确数 = 左边 0 的个数 + 右边 1 的个数
        # 左边 0 的个数 = i - (左边 1 的个数) = i - cumsum[i-1]（i>0 时）
        # 右边 1 的个数 = total_ones - cumsum[i-1]
        left_zeros = np.zeros(N, dtype=int)
        right_ones = np.zeros(N, dtype=int)

        # i = 0 的情况（阈值取最小值）
        left_zeros[0] = 0
        right_ones[0] = total_ones

        # i >= 1 的情况（利用数组切片一次性计算）
        left_zeros[1:] = np.arange(1, N) - cumsum[:-1]   # i - cumsum[i-1]
        right_ones[1:] = total_ones - cumsum[:-1]

        correct = left_zeros + right_ones   # 每个位置作为分割点时正确分类的样本数

        # 6. 只保留特征值发生变化的位置作为候选阈值（与原代码用 np.unique 的语义一致）
        # 特征值重复时，第一个出现的位置才是一个有效的不同阈值
        is_unique_start = np.concatenate(([True], x_sorted[1:] != x_sorted[:-1]))
        unique_indices = np.where(is_unique_start)[0]

        # 取出候选阈值及其对应的正确数
        candidate_thresholds = x_sorted[unique_indices]
        candidate_correct = correct[unique_indices]

        # 7. 找到正确数最大的阈值（若并列，取第一个，与原循环行为一致）
        best_idx = np.argmax(candidate_correct)
        self.threshold = candidate_thresholds[best_idx]
        # 注：原代码没有保存 max_acc，这里也不需要额外存储
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Predicts binary class (0=Star, 1=Non-Star) using the learned threshold.

        Args:
            X: Feature array of shape (M, D).

        Returns:
            preds: Array of binary predictions (0 or 1) of shape (M,).
        """
        # Return array of predictions (0 or 1)
        return (X[:,self.feature_idx]>=self.threshold).astype(np.uint8)


## 第 3 节 —— K 近邻（KNN）

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/e/e7/KnnClassification.svg" width="400">
<br>
<em>(KNN classification logic. Source: Wikimedia Commons)</em>
</center>

树桩只能切一刀；要把 GALAXY 和 QSO 分开，我们需要真正的多分类模型。**K 近邻**是最直观的一个：要判断一个点，就找出离它最近的 $K$ 个训练样本，然后多数投票。

距离用欧氏距离：

$$d(x, y) = \sqrt{\sum_{i=1}^{D}(x_i - y_i)^2}$$

但我们并不真的开方 —— 开方是单调的，按平方距离排序结果一样。把平方展开后可以用一次矩阵乘法搞定：

$$\sum_{i=1}^{D}(x_i - y_i)^2 = \sum_{i=1}^{D}x_i^2 + \sum_{i=1}^{D}y_i^2 - 2\sum_{i=1}^{D}x_i y_i$$

这正是实现里先预计算 `X_train_square`、`X_test_square`，之后就只需 `X_batch @ X_train.T` 的原因。两个工程细节让它可用：距离矩阵**分块**计算，避免 $(M \times N)$ 数组撑爆内存；`np.argpartition` 以线性时间选出最小的 $k$ 个距离，不需要完整排序。


In [ ]:
class KNN:
    # You can change the k value to see the prediction performance differences!
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """
        Lazy learning: just store the data.

        Args:
            X: Training features of shape (N, D).
            y: Training labels of shape (N,).
        """
        self.X_train = X
        self.y_train = y

    def predict(self, X_test: np.ndarray) -> np.ndarray:
      """
      Predicts labels for test data using chunking to avoid memory blow.
      """
      # 预计算训练样本的平方和 (N,)
      X_train_square = np.sum(np.square(self.X_train), axis=1)
      # 预计算测试样本的平方和 (M,)
      X_test_square = np.sum(np.square(X_test), axis=1)

      M = X_test.shape[0]
      chunk_size = 2048          # 根据你的RAM调整，2048通常安全
      predictions = []

      for start in range(0, M, chunk_size):
          end = min(start + chunk_size, M)
          X_batch = X_test[start:end]                # (batch_size, D)
          batch_square = X_test_square[start:end]    # (batch_size,)

          # 计算该批次与所有训练样本的距离平方矩阵 (batch_size, N)
          multiply = X_batch @ self.X_train.T        # (batch_size, N)
          dist_sq = batch_square[:, np.newaxis] + X_train_square[np.newaxis, :] - 2 * multiply

          # 找到每个测试样本最近的k个邻居的索引 (batch_size, k)
          nearest_indices = np.argpartition(dist_sq, kth=self.k-1, axis=-1)[:, :self.k]
          # 注意：argpartition 不保证完全排序，但前k个是k个最小的（无序）
          # 如果需要严格排序（多数投票不要求顺序），argpartition足够；若要按距离排序可用argsort但稍慢。

          # 根据邻居索引获取标签，并进行多数投票
          batch_preds = np.array([
              np.argmax(np.bincount(self.y_train[indices]))
              for indices in nearest_indices
          ])
          predictions.append(batch_preds)

      return np.concatenate(predictions)


## 第 4 节 —— K-Means 聚类

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/e/ea/K-means_convergence.gif" width="400">
<br>
<em>(K-Means iterative process. Source: Wikimedia Commons)</em>
</center>

前面的模型都用到了标签。**K-Means** 完全不用标签：它在特征空间里寻找 $K$ 个自然分组，让我们看看数据本身是否“愿意”分成三类。

$$\text{WCSS} = \sum_{i=1}^{N} \min_{j} \lVert x^{(i)} - \mu^{(j)} \rVert^2$$

算法交替执行两个向量化步骤 —— 把每个点分给最近的质心，再把每个质心移到其所属点的均值 —— 直到质心不再移动。空簇则保留原质心。

选 $K$ 用**肘部法**：把组内平方和（WCSS）对 $K$ 作图，取那个“肘部”。为了不靠肉眼，`find_optimal_k` 直接挑出到首尾两点连线**垂直距离最大**的那个点。


In [ ]:
class KMeans:
    def __init__(self, k=3, max_iters=100, seed=42):
        self.k = k
        self.max_iters = max_iters
        self.seed = seed
        self.centroids = None
        self.wcss = 0.0

    def fit(self, X: np.ndarray) -> Tuple[np.ndarray, float]:
        """
        Runs K-Means clustering.

        Args:
            X: Input data array of shape (N, D).

        Returns:
            clusters: Array of shape (N,) containing cluster indices.
            wcss: The Within-Cluster Sum of Squares.
        """
        np.random.seed(self.seed)
        N, D = X.shape
        # Initialize centroids (randomly)
        self.centroids=X[np.random.choice(N,self.k,replace=False)]
        centroid=self.centroids.copy()
        # Initialize clusters
        clusters = np.zeros(N, dtype=int)
        iter=1
        while iter<=self.max_iters:
        # Loop until convergence or max_iters:
            # 1. Assign clusters (vectorized distance)
            centroid=self.centroids.copy()
            C_sq=np.sum(np.square(self.centroids),axis=-1) #(self.k,)
            X_sq=np.sum(np.square(X),axis=-1) #(N,)
            multiply=X.dot(self.centroids.T)
            distance_squared=C_sq[np.newaxis,:]+X_sq[:,np.newaxis]-2*multiply
            if iter==1:
              clusters=np.argmin(distance_squared,axis=-1)
            else:
              old_clusters=clusters.copy()
              clusters=np.argmin(distance_squared,axis=-1)
            # 2. Update centroids (mean)
            one_hot=np.eye(self.k)[clusters]
            if iter==1:
              counts=np.sum(one_hot,axis=0)
            else:
              old_counts=counts.copy()
              counts=np.sum(one_hot,axis=0)
            sums=np.dot(one_hot.T,X)
            centroid[counts>0]=sums[counts>0]/counts[counts>0,np.newaxis]
            # 3. Check for convergence
            if np.allclose(centroid, self.centroids):
              break
            else:
              iter+=1
              self.centroids=centroid.copy()
        # Calculate final WCSS
        self.wcss=np.sum(np.square(np.abs(X-self.centroids[clusters]))) # Calculate real WCSS here

        return clusters, self.wcss


def find_optimal_k(X: np.ndarray, k_range: range) -> Tuple[int, np.ndarray]:
    """
    Finds optimal K by scanning a range and calculating the distance.
    You can use loop here, we don't detect in this function.
    You can assume that the len(k_range) >= 3 to make this function meaningful

    Args:
        X: Input data.
        k_range: List/Range of K values to test.

    Returns:
        optimal_k: The estimated optimal K.
        wcss_history: List of WCSS values for each K.
    """
    wcss_history = []

    # 1. Iterate over k_range
    for k in k_range:
    # 2. Fit KMeans and collect WCSS
      kmeans=KMeans(k=k)
      cluster,wcss=kmeans.fit(X)
      wcss_history.append(wcss)
    # 3. Calculate the distance of each point
    wcss_history=np.array(wcss_history)
    k_array=np.array(list(k_range))
    x1,y1=k_range[0],wcss_history[0]
    x2,y2=k_range[-1],wcss_history[-1]
    if (x2-x1)**2+(y2-y1)**2!=0:
      geometric_distance=np.abs((x2-x1)*(y1-wcss_history)-(x1-k_array)*(y2-y1))/np.sqrt((x2-x1)**2+(y2-y1)**2)
      optimal_k = k_range[np.argmax(geometric_distance)]
    else:
      return k_array[0],wcss_history

    return optimal_k, wcss_history


def plot_WCSS(optimal_k: int, wcss_history: np.ndarray) -> None:
    """
    Plots WCSS vs K and highlights the selected optimal K.

    Args:
        optimal_k: The optimal K returned by find_optimal_k.
        wcss_history: WCSS values ordered by K=1..len(wcss_history).
    """
    # 1. Build x-axis K values from wcss_history length
    xaxis=list(range(1,len(wcss_history)+1))
    # 2. Plot WCSS curve with marker='o'
    plt.plot(xaxis,wcss_history,marker='o')
    # 3. Set title/xlabel/ylabel and enable grid
    plt.title('Elbow Method: WCSS vs K')
    plt.xlabel('Number of Clusters (K)')
    plt.ylabel('WCSS')
    plt.grid()
    # 4. Highlight optimal_k with a vertical line or distinct marker
    plt.plot([optimal_k],[wcss_history[optimal_k-1]],marker='x')


## 第 5 节 —— 逐类别评估指标

<center>
<svg width="520" height="240" viewBox="0 0 520 240" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Confusion matrix layout">
  <rect x="0" y="0" width="520" height="240" fill="white"/>
  <text x="280" y="24" font-size="16" text-anchor="middle" fill="#2F3B52">Predicted label</text>
  <text x="28" y="140" font-size="16" text-anchor="middle" fill="#2F3B52" transform="rotate(-90 28 140)">True label</text>
  <rect x="140" y="50" width="280" height="160" fill="none" stroke="#2F3B52" stroke-width="2"/>
  <line x1="280" y1="50" x2="280" y2="210" stroke="#2F3B52" stroke-width="2"/>
  <line x1="140" y1="130" x2="420" y2="130" stroke="#2F3B52" stroke-width="2"/>
  <text x="210" y="92" font-size="18" text-anchor="middle" fill="#4C78A8">TP</text>
  <text x="350" y="92" font-size="18" text-anchor="middle" fill="#F58518">FP</text>
  <text x="210" y="172" font-size="18" text-anchor="middle" fill="#F58518">FN</text>
  <text x="350" y="172" font-size="18" text-anchor="middle" fill="#4C78A8">TN</text>
  <text x="210" y="44" font-size="13" text-anchor="middle" fill="#2F3B52">pred=1</text>
  <text x="350" y="44" font-size="13" text-anchor="middle" fill="#2F3B52">pred=0</text>
  <text x="132" y="92" font-size="13" text-anchor="end" fill="#2F3B52">true=1</text>
  <text x="132" y="172" font-size="13" text-anchor="end" fill="#2F3B52">true=0</text>
</svg>
<br>
<em>(Confusion Matrix: The basis for Precision and Recall)</em>
</center>

当一个类别占了 65% 的数据时，准确率会严重误导，所以我们要逐类别看。对类别 $C$：

$$\text{Precision}_C = \frac{TP}{TP + FP + \epsilon}, \qquad \text{Recall}_C = \frac{TP}{TP + FN + \epsilon}, \qquad \text{F1}_C = 2\,\frac{\text{Precision}_C \cdot \text{Recall}_C}{\text{Precision}_C + \text{Recall}_C + \epsilon}$$

整个混淆矩阵只用一次 `np.bincount` 作用在展平索引 `true * C + pred` 上即可得到，无需对类别写 Python 循环。Recall 正是开头那个平衡准确率的组成单元，所以我们才自己算这些数字，而不是直接 import。


In [ ]:
def evaluate_per_class(y_true: np.ndarray, y_pred: np.ndarray, class_names: List[str]) -> pd.DataFrame:
    """
    Computes Precision, Recall, and F1 for each class.

    Args:
        y_true: True labels of shape (N,).
        y_pred: Predicted labels of shape (N,).
        class_names: List of class names (e.g., ['STAR', 'GALAXY', 'QSO']).

    Returns:
        metrics_df: A DataFrame where Index=Class and Columns=[Precision, Recall, F1].
    """
    metrics = []
    epsilon = 1e-9
    num_classes = len(class_names)

    # Vectorized Metric Calculation (No explicit class loop needed for logic, mostly)

    # 1. Compute Confusion Matrix (C x C)
    # Rows=True, Cols=Pred
    # Trick: Use bincount on flattened index (true * C + pred)
    flatten_idx=y_true*num_classes+y_pred
    confusion_matrix=np.bincount(flatten_idx,minlength=num_classes**2).reshape((num_classes,num_classes))
    print(confusion_matrix)
    # 2. Extract TP, FP, FN
    TP=confusion_matrix[np.arange(num_classes),np.arange(num_classes)]
    FP=np.sum(confusion_matrix,axis=0)-TP
    FN=np.sum(confusion_matrix,axis=1)-TP
    # 3. Compute Metrics
    Precision=TP/(TP+FP+epsilon)
    Recall=TP/(TP+FN+epsilon)
    F1=2*(Precision*Recall)/(Precision+Recall+epsilon)
    # 4. Format Output
    metrics=np.stack((Precision,Recall,F1),axis=-1)

    return pd.DataFrame(metrics, index=class_names, columns=['Precision', 'Recall', 'F1'])


## 把经典模型串起来

现在把整套手写工具箱放到真实的比赛数据上跑一遍：读取 `train.csv`、预处理、看 redshift 分布、拟合决策树桩、训练一个 KNN 分类器，最后打印逐类别指标。

两点说明：

* 手写 KNN 要把每个测试点与每个训练点比较，在全部 57.7 万行上拟合内存远远不够。因此我们在训练数据的一个随机子集上拟合，再在该子集的留出部分上评估。`test.csv` 没有标签，不能用来评分。
* 这个基线是用来**理解**的；真正参与比赛角逐的模型从第 6 节开始。


In [ ]:
# =====================================================================
# Classic baseline on the competition data
# =====================================================================
train_path = find_csv("train.csv")
train_df = pd.read_csv(train_path)
print("train:", train_df.shape)

# Statistics are computed on the training frame only (no data leakage).
X_train, y_train, _ = preprocess_data(train_df)

# --- Look at the redshift distribution --------------------------------
plot_redshift_distribution(train_df)
plt.show()

# --- Section 2: the decision stump, on the full training set ----------
print("\n--- Decision stump (STAR vs non-STAR) ---")
stump = DecisionStump()
stump.fit(X_train, y_train)
print(f"optimal redshift threshold: {stump.threshold:.4f}")
print(f"stump accuracy on train: {(stump.predict(X_train) == (y_train != 0).astype(int)).mean():.4f}")

# --- Section 3 & 5: KNN on a subset -----------------------------------
# The from-scratch KNN is O(N_test * N_train); on the full data that is far
# too heavy, so we fit it on a random subset and score it on a held-out part.
rng = np.random.default_rng(42)
idx = rng.permutation(len(X_train))[:20000]
X_sub, y_sub = X_train[idx], y_train[idx]
n_fit = 16000

print("\n--- K-nearest neighbours (k = 100) ---")
knn = KNN(k=100)
knn.fit(X_sub[:n_fit], y_sub[:n_fit])
y_pred = knn.predict(X_sub[n_fit:])
print(f"KNN accuracy on the held-out subset: {(y_pred == y_sub[n_fit:]).mean():.4f}")

print("\n--- Per-class metrics (KNN) ---")
display(evaluate_per_class(y_sub[n_fit:], y_pred, ["STAR", "GALAXY", "QSO"]))

# --- Section 4: the K-means elbow, on a subset ------------------------
print("\n--- K-means: searching for the optimal K ---")
sub = X_train[rng.permutation(len(X_train))[:5000]]
optimal_k, wcss_history = find_optimal_k(sub, range(1, 7))
plot_WCSS(optimal_k, wcss_history)
plt.show()
print(f"optimal K detected by the elbow method: {optimal_k}")


## 深度模型（第 6–14 节）

第 1–5 节我们用很小的特征集**手写**了经典算法，为的是让数学保持清晰易读。从这里开始，我们卸下辅助轮：可以使用 `scikit-learn` 与 `PyTorch`，并且直接优化真正的比赛指标。

| | 第 1–5 节 | 第 6–14 节 |
| :--- | :--- | :--- |
| **目标** | 理解算法 | 尽量提高排行榜分数 |
| **库** | 只用 `numpy` / `pandas` | `scikit-learn`、`PyTorch` 等 |
| **特征** | 8 个，便于理解数学 | 14 个，为精度而工程化 |

第 6 节搭好后续共用的全部基础设施，第 7 节建立一个强力的经典基线，第 8–12 节依次介绍 MLP、RNN、LSTM、Transformer 与 GAN，第 13 节把它们融合成最终的 `submission.csv`。


## 第 6 节 —— 比赛环境：指标、划分与共享工具

### 6.1 无泄漏的验证集

我们无法在测试集上调参（它的标签我们看不到）。因此我们**从训练数据中切出一份验证集**，并使用*分层*划分，使 65 / 20 / 14 % 的类别比例在两半中都保持不变。

同时遵守与第 1 节相同的防泄漏规则：**所有归一化统计量（min / max）只在训练那一半上计算**，再应用到验证集与测试集。若在整个训练集上计算，就会把验证集信息泄漏进模型 —— 这是一个很容易犯、又很难发现的错误。

### 6.2 特征工程 —— 颜色指数

五个星等 `u, g, r, i, z` 表示天体在五个不同颜色滤光片下的亮度。但天体的*绝对*亮度还取决于它离我们有多远。真正决定一个天体是恒星、星系还是类星体的，是它的**颜色**，即相邻两个波段的*差值*：

$$u-g,\qquad g-r,\qquad r-i,\qquad i-z$$

这四个**颜色指数**正是天文学家实际使用的*颜色-颜色图*的坐标。把它们喂给模型，是*用领域知识做特征工程*的经典范例 —— 计算便宜，却常常比换一个更花哨的模型更值钱。

我们同时保留两个类别列（`spectral_type`、`galaxy_population`）的小整数编码。最终得到 **14 个特征**：`alpha, delta, u, g, r, i, z, redshift` + 2 个类别编码 + 4 个颜色指数。

### 6.3 类别权重

为了对抗类别不均衡，我们让每个类别的权重与其频率成反比：

$$w_c = \frac{N}{C \cdot n_c}$$

这样，在稀有的 STAR 上犯一次错，代价大约是常见的 GALAXY 的 $2.32$ 倍。这些权重会直接进入损失函数（第 8–11 节）。

### 6.4 共享训练工具

这一部分里每个神经网络都用*同一套*训练循环，所以我们只写一次：

* `train_torch(...)` —— mini-batch 训练循环，内含 **AdamW**、**One-Cycle 学习率调度**、类别加权的**交叉熵**，并按**平衡准确率挑选最佳权重**；
* `predict_proba(...)` —— 分批计算的 softmax 概率（后面做集成时要用）；
* `balanced_accuracy(...)` / `report(...)` —— 我们自己的指标工具。

开头的两个旋钮（`NN_TRAIN_SIZE`、`*_EPOCHS`）让你在时间与精度之间做取舍。默认值是按免费的 Colab GPU 来设定的。


In [ ]:
# =====================================================================
# Section 6 - Competition setup
# =====================================================================
import os
import glob
import copy
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cpu":
    print("WARNING: no GPU found. In Colab use Runtime > Change runtime type > GPU (T4).")

# --- Locate the CSVs (find_csv is defined in the setup cell) -----------
train_path, test_path = find_csv("train.csv"), find_csv("test.csv")
DATA_DIR = os.path.dirname(train_path)
print("DATA_DIR:", DATA_DIR or ".")

# --- Feature definition -------------------------------------------------
CLASS_MAP = {"STAR": 0, "GALAXY": 1, "QSO": 2}
INV_CLASS_MAP = {v: k for k, v in CLASS_MAP.items()}
SPECTRAL_MAP = {"M": 0, "A/F": 1, "O/B": 2, "G/K": 3}
POPULATION_MAP = {"Red_Sequence": 0, "Blue_Cloud": 1}
RAW_FEATURES = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
CATEGORICAL_FEATURES = ["spectral_type_code", "galaxy_population_code"]
COLOR_INDEX_FEATURES = ["u_g", "g_r", "r_i", "i_z"]
FEATURES = RAW_FEATURES + CATEGORICAL_FEATURES + COLOR_INDEX_FEATURES

def add_engineered_features(df):
    df = df.copy()
    df["spectral_type_code"] = df["spectral_type"].map(SPECTRAL_MAP).astype("float64")
    df["galaxy_population_code"] = df["galaxy_population"].map(POPULATION_MAP).astype("float64")
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]
    return df

def build_features(df, feature_cols, stats=None):
    """Encode, engineer and min-max normalise. Stats must come from the training split only."""
    df = add_engineered_features(df)
    if stats is None:
        stats = {c: {"min": float(df[c].min()), "max": float(df[c].max())} for c in feature_cols}
    for c in feature_cols:
        span = stats[c]["max"] - stats[c]["min"]
        df[c] = (df[c] - stats[c]["min"]) / span if span != 0 else 0.0
    X = df[feature_cols].to_numpy(dtype="float32")
    y = df["class"].map(CLASS_MAP).to_numpy() if "class" in df.columns else None
    return X, y, stats

# --- Load and split -----------------------------------------------------
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print("train:", train_df.shape, " test:", test_df.shape)

tr_df, va_df = train_test_split(train_df, test_size=0.2, random_state=SEED, stratify=train_df["class"])
X_tr, y_tr, STATS = build_features(tr_df, FEATURES)
X_va, y_va, _ = build_features(va_df, FEATURES, stats=STATS)
X_te, _, _ = build_features(test_df, FEATURES, stats=STATS)
print("X_tr", X_tr.shape, "| X_va", X_va.shape, "| X_te", X_te.shape)

# --- Class weights for the imbalanced loss ------------------------------
counts = np.bincount(y_tr, minlength=3)
class_weights = torch.tensor(len(y_tr) / (3.0 * counts), dtype=torch.float32, device=DEVICE)
for i, name in enumerate(["STAR", "GALAXY", "QSO"]):
    print(f"  {name:7s} n={counts[i]:7d}  weight={class_weights[i]:.3f}")

# --- Metric helpers -----------------------------------------------------
def balanced_accuracy(y_true, y_pred, n_classes=3):
    recalls = [float((y_pred[y_true == c] == c).mean()) if (y_true == c).any() else 0.0
               for c in range(n_classes)]
    return float(np.mean(recalls))

def report(name, y_true, y_pred):
    print(f"{name:14s} accuracy={accuracy_score(y_true, y_pred):.4f}   "
          f"balanced_acc={balanced_accuracy(y_true, y_pred):.4f}   "
          f"macro_f1={f1_score(y_true, y_pred, average='macro'):.4f}")

@torch.no_grad()
def predict_proba(model, X, batch_size=8192):
    """Batched softmax probabilities for any torch classifier."""
    model.eval()
    probs = []
    for i in range(0, len(X), batch_size):
        xb = torch.from_numpy(np.ascontiguousarray(X[i:i + batch_size])).to(DEVICE)
        probs.append(torch.softmax(model(xb), dim=-1).cpu())
    return torch.cat(probs).numpy()

# --- The shared training loop ------------------------------------------
def train_torch(model, X_tr_, y_tr_, X_va_, y_va_, epochs=25, batch_size=1024, lr=2e-3,
                weight_decay=1e-4, label_smoothing=0.0, verbose=True):
    model.to(DEVICE)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(np.ascontiguousarray(X_tr_)),
                      torch.from_numpy(np.asarray(y_tr_)).long()),
        batch_size=batch_size, shuffle=True, drop_last=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=epochs * max(1, len(loader)), pct_start=0.3)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)

    best_score, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            if scheduler.last_epoch < scheduler.total_steps - 1:
                scheduler.step()
            running += loss.item() * len(xb)
        pred = predict_proba(model, X_va_).argmax(1)
        score = balanced_accuracy(y_va_, pred)
        if score > best_score:
            best_score, best_state = score, copy.deepcopy(model.state_dict())
        if verbose and (epoch % 5 == 0 or epoch == epochs):
            print(f"  epoch {epoch:02d}  loss={running / len(loader.dataset):.4f}  "
                  f"val_balanced_acc={score:.4f}")
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"  -> best val_balanced_acc = {best_score:.4f}")
    return model

# --- Knobs (lower them if you are in a hurry / on CPU) ------------------
NN_TRAIN_SIZE = 200_000
MLP_EPOCHS = 30
SEQ_EPOCHS = 25
GAN_STEPS = 2000
print("\nSetup ready.")


## 第 7 节 —— 强力的经典基线：梯度提升树

在伸手去拿神经网络之前，专业习惯是先建立一个**强基线**。对于*表格型*数据（行 × 列，没有图像、没有音频），最强的经典家族是**梯度提升决策树**。

### 三步理解它

1. **决策树**用类似“*redshift 是否小于 0.0016？*”的问题来切分数据。单棵树是一个很弱、方差很大的模型。
2. **提升（boosting）**把弱变强：先拟合一棵树，看看它在哪些样本上出错，然后针对这些错误再拟合*下一棵*树。每棵新树都在修正之前的集成。几百轮之后，你就得到一个非常准的模型。
3. **直方图式**训练（`HistGradientBoostingClassifier` 的做法）把每个特征分到 ≤ 255 个桶里，这让它在 50 万行数据上又快又省内存。

### 为什么要 `class_weight="balanced"`？

记住指标：*平衡*准确率。因为有了类别权重，树漏掉一个 STAR 会被罚得更重，于是模型会倾向于预测稀有类别 —— 这正是指标所奖励的。你会看到准确率略微**下降**，而平衡准确率**上升**：这说明指标在起作用。

### 怎么看输出

* 打印的表格对比了 `accuracy`、`balanced_acc` 与 `macro_f1`。
* `permutation_importance` 一次打乱一个特征，观察分数掉了多少。这告诉我们*哪个物理量最重要* —— 你应该看到 `redshift` 排在最前，其次是颜色指数，正好与第 2 节的天文学直觉相互印证。


In [ ]:
# =====================================================================
# Section 7 - Gradient-boosted trees baseline
# =====================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

t0 = time.time()
gbdt = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.08,
    max_leaf_nodes=63,
    class_weight="balanced",   # optimise the metric that Kaggle actually uses
    random_state=SEED,
)
gbdt.fit(X_tr, y_tr)
proba_gbdt = gbdt.predict_proba(X_va)
report("GBDT baseline", y_va, proba_gbdt.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")

# --- Which features carry the signal? -----------------------------------
sub = np.random.choice(len(X_va), size=min(20000, len(X_va)), replace=False)
perm = permutation_importance(gbdt, X_va[sub], y_va[sub], n_repeats=3,
                              random_state=SEED, scoring="balanced_accuracy", n_jobs=-1)
imp = (pd.DataFrame({"feature": FEATURES, "importance": perm.importances_mean})
         .sort_values("importance", ascending=False).reset_index(drop=True))
display(imp)

plt.figure(figsize=(8, 5))
plt.barh(imp["feature"][::-1], imp["importance"][::-1])
plt.xlabel("Drop in balanced accuracy when the feature is shuffled")
plt.title("Permutation Importance (Gradient-Boosted Trees)")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## 第 8 节 —— 多层感知机（MLP）

深度学习由此开始。MLP 是神经网络的 “hello world”，而且在表格数据上，它也是我们要见到的五种结构里最强的一种。

### 8.1 单个神经元

一个神经元做的是**仿射变换 + 非线性**：

$$z = \mathbf{w}^\top \mathbf{x} + b, \qquad a = \sigma(z)$$

* $\mathbf{w}$ 是**权重**（需要学习），$b$ 是**偏置**（也需要学习）；
* $\sigma$ 是**激活函数**。没有它，多层叠加会塌缩成一个大的线性映射 —— 而线性模型无法把星系和类星体分开。

### 8.2 从神经元到网络

MLP 把神经元堆成**层**。第 $\ell$ 层根据上一层的激活值产生新的激活值：

$$\mathbf{a}^{(\ell)} = \sigma\!\left(\mathbf{W}^{(\ell)}\mathbf{a}^{(\ell-1)} + \mathbf{b}^{(\ell)}\right)$$

最后一层输出 **3 个数（logits）**，每个类别一个。**softmax** 把它们变成概率：

$$\hat{p}_c = \frac{e^{z_c}}{\sum_{j} e^{z_j}}$$

### 8.3 它是怎么学习的 —— 反向传播 + 梯度下降

1. **前向传播**：把一个小批量数据推过网络，得到概率。
2. **损失**：用**交叉熵**与真值比较（并带上我们的类别权重 $w_c$）：

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^{B} w_{y_i}\,\log \hat{p}_{i,\,y_i}$$

3. **反向传播**：autograd 应用链式法则，给出每个参数 $\theta$ 的 $\partial\mathcal{L}/\partial\theta$。
4. **更新**：让每个参数沿梯度反方向走一小步，$\theta \leftarrow \theta - \eta\, \nabla_\theta\mathcal{L}$，然后重复。

这正是 `train_torch` 内部的那个循环。

### 8.4 我们的结构

```
input(14) -> Linear(256) -> BatchNorm -> ReLU -> Dropout
          -> Linear(256) -> BatchNorm -> ReLU -> Dropout
          -> Linear(128) -> BatchNorm -> ReLU -> Dropout
          -> Linear(3)
```

* **BatchNorm** 重新中心化每层的输入，使训练更稳定、更快。
* **Dropout** 在训练时随机把 20% 的激活值置零，强迫网络不要依赖任何单个单元 —— 一种便宜又有效的正则化。
* **AdamW** 是带逐参数自适应步长、并解耦了权重衰减的梯度下降变体，是业界的默认选择。
* **One-Cycle 学习率**：学习率先升后降，训练更快，也更容易落到好的极小值。

> **要记住的关键思想：** 以上一切都是*学出来的*。我们从没告诉它 redshift 重要 —— 是它自己发现的。


In [ ]:
# =====================================================================
# Section 8 - Multi-Layer Perceptron
# =====================================================================
class MLP(nn.Module):
    """A plain feed-forward network: Linear -> BatchNorm -> ReLU -> Dropout, stacked."""
    def __init__(self, n_features, hidden=(256, 256, 128), n_classes=3, dropout=0.2):
        super().__init__()
        layers, prev = [], n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Subsample the training set so that one epoch stays cheap; raise NN_TRAIN_SIZE for more accuracy.
idx = np.random.choice(len(X_tr), size=min(NN_TRAIN_SIZE, len(X_tr)), replace=False)
X_nn, y_nn = X_tr[idx], y_tr[idx]
print("neural-net training set:", X_nn.shape)

t0 = time.time()
mlp = MLP(X_nn.shape[1])
mlp = train_torch(mlp, X_nn, y_nn, X_va, y_va, epochs=MLP_EPOCHS, batch_size=4096, lr=3e-3)
proba_mlp = predict_proba(mlp, X_va)
report("MLP", y_va, proba_mlp.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")


## 第 9 节 —— 循环神经网络（RNN）

### 9.1 什么时候用 RNN？

RNN 是为**序列**设计的：句子里的词、视频里的帧、随时间变化的价格。它的标志性特征是**隐状态** $\mathbf{h}_t$，它会在每一步之间传递，充当“已经看过的一切”的滚动记忆。

我们的数据是表格而不是句子 —— 但我们仍然可以把它*看成*序列：把 14 个特征各自当作一个**词元（token）**，逐个读入。这一半是教学上的构造，而“表格 Transformer”“特征词元 RNN”在实际中正是这么做的。

### 9.2 循环公式

在第 $t$ 步，单元接收词元 $\mathbf{x}_t$（这里把标量投影成向量）和上一步的隐状态：

$$\mathbf{h}_t = \tanh\!\left(\mathbf{W}_{x}\mathbf{x}_t + \mathbf{W}_{h}\,\mathbf{h}_{t-1} + \mathbf{b}\right)$$

* $\mathbf{W}_x$ 混合**新输入**，$\mathbf{W}_h$ 混合**记忆**；
* **每一步都复用同一组权重** —— 这种权重共享让 RNN 用固定数量的参数处理任意长度的输入；
* 读完最后一个词元后，$\mathbf{h}_{T}$ 就是整行数据的概括，交给一个小分类头即可。

### 9.3 梯度消失问题

训练 RNN 意味着要*沿时间*反向传播：

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_h} \sim \prod_{t} \mathbf{W}_h^\top \,\mathrm{diag}\!\left(1-\mathbf{h}_t^2\right)$$

如果这些因子小于 1，乘积会随序列长度指数级缩小 —— 梯度**消失**，网络就忘了前面的词元；若大于 1，则会爆炸。第 10 节用门控解决了前一个问题。

### 9.4 代码形式

因为我们希望你*看见*这个循环，我们手写这个循环，而不是调用 `nn.RNN`：

```python
h = torch.zeros(batch, hidden)
for t in range(n_features):
    h = torch.tanh(Wx(tokens[:, t]) + Wh(h))
```

那个 `for` 循环**就是** RNN。其余部分（标量特征的嵌入、分类头、训练）都只是管道。


In [ ]:
# =====================================================================
# Section 9 - Recurrent Neural Network (hand-written recurrence)
# =====================================================================
class TabularRNN(nn.Module):
    """Reads the D features one at a time and keeps a single hidden state."""
    def __init__(self, n_features, d_model=64, hidden=128, n_classes=3, dropout=0.2):
        super().__init__()
        self.n_features = n_features
        self.embed = nn.Linear(1, d_model)          # scalar feature -> vector token
        self.wx = nn.Linear(d_model, hidden)        # input -> hidden
        self.wh = nn.Linear(hidden, hidden, bias=False)  # hidden -> hidden
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Dropout(dropout),
                                  nn.Linear(hidden, n_classes))

    def forward(self, x):
        h = torch.zeros(x.size(0), self.wh.out_features, device=x.device)
        tokens = self.embed(x.unsqueeze(-1))        # (B, D, d_model)
        for t in range(self.n_features):
            h = torch.tanh(self.wx(tokens[:, t]) + self.wh(h))
        return self.head(h)

t0 = time.time()
rnn = TabularRNN(X_nn.shape[1])
rnn = train_torch(rnn, X_nn, y_nn, X_va, y_va, epochs=SEQ_EPOCHS, batch_size=1024, lr=2e-3)
proba_rnn = predict_proba(rnn, X_va)
report("RNN", y_va, proba_rnn.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")


## 第 10 节 —— 长短期记忆网络（LSTM）

LSTM（Hochreiter & Schmidhuber, 1997）用一个**细胞状态** $\mathbf{c}_t$ 取代了普通的隐状态。细胞状态像一条“传送带”，由三个**门**控制，从而解决了梯度消失。

### 10.1 三个门

门就是一个 sigmoid 层，输出落在 $(0,1)$：0 表示“完全阻断”，1 表示“全部放行”。

$$
\begin{aligned}
\mathbf{i}_t &= \sigma\!\left(\mathbf{W}_i[\mathbf{x}_t,\mathbf{h}_{t-1}] + \mathbf{b}_i\right) && \text{输入门：写入多少新信息}\\
\mathbf{f}_t &= \sigma\!\left(\mathbf{W}_f[\mathbf{x}_t,\mathbf{h}_{t-1}] + \mathbf{b}_f\right) && \text{遗忘门：从记忆中抹去什么}\\
\mathbf{o}_t &= \sigma\!\left(\mathbf{W}_o[\mathbf{x}_t,\mathbf{h}_{t-1}] + \mathbf{b}_o\right) && \text{输出门：对外暴露什么}\\
\tilde{\mathbf{c}}_t &= \tanh\!\left(\mathbf{W}_c[\mathbf{x}_t,\mathbf{h}_{t-1}] + \mathbf{b}_c\right) && \text{候选记忆}
\end{aligned}
$$

然后是状态更新：

$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t, \qquad \mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

### 10.2 为什么它有用

看细胞状态那一行公式。从 $\mathbf{c}_t$ 回到 $\mathbf{c}_{t-1}$ 的梯度路径乘的是 $\mathbf{f}_t$ —— 一个*学出来的*门，而不是固定的矩阵。因此网络可以学着让 $\mathbf{f}_t \approx 1$，把信息跨很多步保留下来。这一条加法通路就是全部的诀窍。

> **实现小技巧。** 我们不用四个独立的 `nn.Linear`，而是用**一个** `nn.Linear(d_model, 4*hidden)`，再用 `chunk` 切成四个门。数学上完全等价，但一次大的矩阵乘法在 GPU 上更快 —— 这个习惯值得抄下来。


In [ ]:
# =====================================================================
# Section 10 - LSTM (hand-written gates)
# =====================================================================
class TabularLSTM(nn.Module):
    """Same sequence view as the RNN, but with an input/forget/output gated cell state."""
    def __init__(self, n_features, d_model=64, hidden=128, n_classes=3, dropout=0.2):
        super().__init__()
        self.n_features, self.hidden = n_features, hidden
        self.embed = nn.Linear(1, d_model)
        self.x2g = nn.Linear(d_model, 4 * hidden)       # input  -> 4 gates
        self.h2g = nn.Linear(hidden, 4 * hidden, bias=False)  # hidden -> 4 gates
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Dropout(dropout),
                                  nn.Linear(hidden, n_classes))

    def forward(self, x):
        B = x.size(0)
        h = torch.zeros(B, self.hidden, device=x.device)
        c = torch.zeros(B, self.hidden, device=x.device)
        tokens = self.embed(x.unsqueeze(-1))
        for t in range(self.n_features):
            gates = self.x2g(tokens[:, t]) + self.h2g(h)          # (B, 4*hidden)
            i, f, g, o = gates.chunk(4, dim=-1)
            i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
            g = torch.tanh(g)
            c = f * c + i * g          # cell state: forget old + write new
            h = o * torch.tanh(c)      # hidden state: gated read-out
        return self.head(h)

t0 = time.time()
lstm = TabularLSTM(X_nn.shape[1])
lstm = train_torch(lstm, X_nn, y_nn, X_va, y_va, epochs=SEQ_EPOCHS, batch_size=1024, lr=2e-3)
proba_lstm = predict_proba(lstm, X_va)
report("LSTM", y_va, proba_lstm.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")


## 第 11 节 —— Transformer

《Attention Is All You Need》（Vaswani 等，2017）用**注意力**取代了循环：不再一步一步传递记忆，而是让每个词元*并行地*看向所有其他词元，并自己决定该把注意力放在哪里。

### 11.1 缩放点积注意力

每个词元被投影成三个向量 —— **Query**（我在找什么）、**Key**（我能提供什么）、**Value**（我实际装着什么）：

$$\mathbf{Q} = \mathbf{X}\mathbf{W}_Q,\qquad \mathbf{K} = \mathbf{X}\mathbf{W}_K,\qquad \mathbf{V} = \mathbf{X}\mathbf{W}_V$$

注意力就是

$$\mathrm{Attn}(\mathbf{Q},\mathbf{K},\mathbf{V}) = \mathrm{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}$$

* $\mathbf{Q}\mathbf{K}^\top$ 是任意两个词元之间的**相似度矩阵**：词元 $i$ 有多在意词元 $j$；
* 除以 $\sqrt{d_k}$ 是为了在头维度变大时不让 logits 爆掉（否则 softmax 会饱和、梯度会死掉）；
* 乘以 $\mathbf{V}$ 得到所有 value 的**加权平均** —— 这就是“注意力”的输出。

### 11.2 多头注意力

一张注意力图只能刻划一种关系。于是我们并行运行 $h$ 个注意力，各自作用在低维投影（“头”）上，再拼接起来：

$$\mathrm{MHA}(\mathbf{X}) = [\mathrm{head}_1;\dots;\mathrm{head}_h]\mathbf{W}_O$$

这就是为什么我们把 (B, T, 3·d_model) 的投影 `reshape` 成 `(3, B, n_heads, T, d_head)` —— 一个很受面试欢迎的 reshape。

### 11.3 Encoder 块

每个块都是标准的“三明治”：

$$\mathbf{X} \leftarrow \mathbf{X} + \mathrm{MHA}(\mathrm{LN}(\mathbf{X})), \qquad \mathbf{X} \leftarrow \mathbf{X} + \mathrm{FFN}(\mathrm{LN}(\mathbf{X}))$$

* **残差连接**（$\mathbf{X} + \dots$）给梯度一条直通底层的捷径 —— 对深网络至关重要；
* **LayerNorm** 稳定每个词元的统计量；
* **FFN** 是逐词元的 MLP（这里是 `Linear → GELU → Linear`），补充非线性处理能力。

### 11.4 位置编码

注意力对排列是不变的：不给额外信息，它分不清第 1 个词元和第 14 个词元。因此我们给每个词元加一个**可学习的位置嵌入**，让模型知道自己在看*哪个*特征。

### 11.5 如何读出类别

Transformer 返回的是一个序列。做分类时我们对词元向量做**平均池化**（另一种做法是引入专门的 `[CLS]` 词元），再接 `LayerNorm → Linear(3)`。

> **诚实的提醒。** 对于 14 个相互独立的表格列，注意力可利用的结构远不如自然语言那么多，所以 Transformer 在这里通常*不会*胜过梯度提升树。我们仍然把它放进来，因为它是现代大语言模型背后的结构，而且多样的模型能组成更好的集成（第 13 节）。


In [ ]:
# =====================================================================
# Section 11 - Transformer encoder (hand-written attention)
# =====================================================================
class MultiHeadSelfAttention(nn.Module):
    """Scaled dot-product attention with several heads in parallel."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model, self.n_heads = d_model, n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, _ = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]                       # (B, heads, T, d_head)
        att = torch.softmax((q @ k.transpose(-2, -1)) / math.sqrt(self.d_head), dim=-1)
        out = (att @ v).transpose(1, 2).reshape(B, T, self.d_model)
        return self.proj(out)

class EncoderBlock(nn.Module):
    """Pre-LN transformer block: attention sub-layer + feed-forward sub-layer."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm1(x)))   # residual + attention
        x = x + self.drop(self.ff(self.norm2(x)))     # residual + MLP
        return x

class TabularTransformer(nn.Module):
    """Each feature is a token; mean-pool the encoder output and classify."""
    def __init__(self, n_features, d_model=64, n_heads=4, d_ff=128, n_layers=2,
                 n_classes=3, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.embed = nn.Linear(1, d_model)
        self.pos = nn.Parameter(torch.zeros(1, n_features, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout)
                                     for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        tokens = self.embed(x.unsqueeze(-1)) + self.pos
        for block in self.blocks:
            tokens = block(tokens)
        return self.head(self.norm(tokens).mean(dim=1))

t0 = time.time()
transformer = TabularTransformer(X_nn.shape[1])
transformer = train_torch(transformer, X_nn, y_nn, X_va, y_va, epochs=SEQ_EPOCHS,
                          batch_size=1024, lr=1.5e-3)
proba_trf = predict_proba(transformer, X_va)
report("Transformer", y_va, proba_trf.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")


## 第 12 节 —— 生成对抗网络（GAN）

到目前为止，每个模型都是*判别式*的：在类别之间划一条边界。而 **GAN**（Goodfellow 等，2014）是*生成式*的：它学习去**创造**数据。

### 12.1 两个玩家的博弈

两个网络互相对抗地训练：

* **生成器** $G$ 接收随机噪声 $\mathbf{z}$，产出一个假样本；
* **判别器** $D$ 看一个样本，必须判断它是真的还是假的。

形式化地，这是一个极小极大博弈：

$$\min_G \max_D \; \mathbb{E}_{\mathbf{x}\sim p_{\text{data}}}\big[\log D(\mathbf{x})\big] + \mathbb{E}_{\mathbf{z}}\big[\log\big(1 - D(G(\mathbf{z}))\big)\big]$$

对 $D$ 做梯度下降（最大化：“识破假货”）与对 $G$ 做梯度下降（最小化：“骗过判别器”）交替进行。在均衡点上，$G$ 复现了真实数据的分布。

### 12.2 条件 GAN

我们的生成器应该生成*指定类别*的样本，所以两个网络还要接收**类别标签**（转成一个小嵌入向量后拼接到输入上）。这就是**条件 GAN**（cGAN）—— 与驱动类别条件图像生成的是同一个技巧。

### 12.3 这为什么帮助我们的指标？

我们的问题是不均衡的：STAR 只占 14%。GAN 让我们能够**制造额外的 STAR 样本**，使它们在统计上看起来像真实样本，把它们加进训练集，再重新训练 MLP。在比赛里这叫**合成数据增强**，是提升平衡准确率的著名手段。

### 12.4 实战警告

* **模式崩塌（mode collapse）** —— 如果 $G$ 找到一个总能骗过 $D$ 的样本，它可能永远只输出那一个样本。要留意生成样本的多样性。
* **$D$ 过强** —— 如果 $D$ 太强，$G$ 的梯度会消失。使用 `betas=(0.5, 0.999)` 和标签平滑（真样本标签用 `0.9` 而不是 `1.0`）是标准的补救办法。
* GAN 并*不是*分类器 —— 它从来看不到指标。它只是让训练这锅汤更浓郁。


In [ ]:
# =====================================================================
# Section 12 - Conditional GAN for minority-class augmentation
# =====================================================================
class Generator(nn.Module):
    """(noise z, class label) -> a plausible feature vector."""
    def __init__(self, n_features, n_classes, z_dim=32, hidden=256):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, 16)
        self.net = nn.Sequential(
            nn.Linear(z_dim + 16, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, n_features))

    def forward(self, z, labels):
        return self.net(torch.cat([z, self.label_emb(labels)], dim=-1))

class Discriminator(nn.Module):
    """(feature vector, class label) -> one real/fake logit."""
    def __init__(self, n_features, n_classes, hidden=256):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, 16)
        self.net = nn.Sequential(
            nn.Linear(n_features + 16, hidden), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(hidden, 1))

    def forward(self, x, labels):
        return self.net(torch.cat([x, self.label_emb(labels)], dim=-1))

Z_DIM = 32
BATCH = 512
generator = Generator(X_nn.shape[1], 3, z_dim=Z_DIM).to(DEVICE)
discriminator = Discriminator(X_nn.shape[1], 3).to(DEVICE)
opt_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

X_pool = torch.from_numpy(np.ascontiguousarray(X_nn))
y_pool = torch.from_numpy(np.asarray(y_nn)).long()

t0 = time.time()
for step in range(GAN_STEPS):
    # ---- 1) Update the Discriminator: real -> 1 (smoothed to 0.9), fake -> 0
    idx = torch.randint(0, len(X_pool), (BATCH,))
    real_x, real_y = X_pool[idx].to(DEVICE), y_pool[idx].to(DEVICE)
    z = torch.randn(BATCH, Z_DIM, device=DEVICE)
    fake_y = y_pool[torch.randint(0, len(y_pool), (BATCH,))].to(DEVICE)
    with torch.no_grad():
        fake_x = generator(z, fake_y)
    opt_d.zero_grad()
    d_loss = (bce(discriminator(real_x, real_y), torch.full((BATCH, 1), 0.9, device=DEVICE))
              + bce(discriminator(fake_x, fake_y), torch.zeros(BATCH, 1, device=DEVICE)))
    d_loss.backward()
    opt_d.step()

    # ---- 2) Update the Generator: it wants the Discriminator to say "1" (real)
    z = torch.randn(BATCH, Z_DIM, device=DEVICE)
    fake_y = y_pool[torch.randint(0, len(y_pool), (BATCH,))].to(DEVICE)
    opt_g.zero_grad()
    g_loss = bce(discriminator(generator(z, fake_y), fake_y), torch.ones(BATCH, 1, device=DEVICE))
    g_loss.backward()
    opt_g.step()

    if (step + 1) % 200 == 0:
        print(f"step {step + 1:5d}  D_loss={float(d_loss):.3f}  G_loss={float(g_loss):.3f}")
print(f"GAN training time: {time.time() - t0:.1f}s")

# --- Compare real vs generated STAR samples on the redshift axis ---------
STAR_IDX = 0
redshift_col = FEATURES.index("redshift")
real_star = X_nn[y_nn == STAR_IDX][:, redshift_col]
with torch.no_grad():
    synth_star = generator(torch.randn(40000, Z_DIM, device=DEVICE),
                           torch.full((40000,), STAR_IDX, dtype=torch.long, device=DEVICE))
synth_star = synth_star.cpu().numpy()

plt.figure(figsize=(9, 4))
plt.hist(real_star, bins=60, alpha=0.5, density=True, label="real STAR")
plt.hist(synth_star[:, redshift_col], bins=60, alpha=0.5, density=True, label="generated STAR")
plt.title("Conditional GAN: real vs generated STAR redshift")
plt.xlabel("Redshift (normalised)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

# --- Augment the minority class and re-train the MLP --------------------
X_aug = np.concatenate([X_nn, synth_star.astype("float32")], axis=0)
y_aug = np.concatenate([y_nn, np.full(len(synth_star), STAR_IDX, dtype=y_nn.dtype)], axis=0)
print("augmented training set:", X_aug.shape,
      "| STAR share: {:.1%}".format((y_aug == STAR_IDX).mean()))

t0 = time.time()
mlp_gan = MLP(X_aug.shape[1])
mlp_gan = train_torch(mlp_gan, X_aug, y_aug, X_va, y_va, epochs=MLP_EPOCHS,
                      batch_size=4096, lr=3e-3)
proba_mlp_gan = predict_proba(mlp_gan, X_va)
report("MLP + GAN aug", y_va, proba_mlp_gan.argmax(1))
print(f"training time: {time.time() - t0:.1f}s")


## 第 13 节 —— 集成与最终提交

我们现在有了六个训练好的模型，它们的归纳偏置确实各不相同：

| 模型 | 它擅长什么 |
| :--- | :--- |
| 梯度提升树 | 轴对齐的阈值切分，能很好地利用强信号 redshift |
| MLP | 光滑的非线性交互 |
| RNN / LSTM | 顺序地阅读各个特征 |
| Transformer | 全局的两两特征关系 |
| MLP + GAN 增强 | 重新平衡后的类别边界 |

当模型犯的是**不同种类的错误**时，对它们的概率求平均一定优于只挑最好的那一个。这就是**集成（ensemble）**思想，也是几乎所有 Kaggle 领奖台作品的收尾方式。

### 做融合，而不只是投票

我们不做硬性的多数投票，而是用权重 $w_m$ 对 **softmax 概率**加权平均：

$$\hat{\mathbf{p}} = \sum_m w_m\, \hat{\mathbf{p}}^{(m)}, \qquad \hat{y} = \arg\max_c \hat{p}_c$$

我们搜索一个小的权重网格，保留**验证集平衡准确率**最高的组合 —— 绝不偷看测试集。

### 轮到你了

* 在 GPU 上试试 `NN_TRAIN_SIZE = len(X_tr)` 以及更多轮次：每个模型都应该更好。
* 把网格搜索换成在验证概率上训练的*逻辑回归*堆叠器（stacker）。
* 融合之后，把获胜模型在 train + validation 上**重新拟合一次**（数据从 80% 变成 100%）再去预测测试集。这是标准做法，通常还能再涨最后零点几个百分点。


In [ ]:
# =====================================================================
# Section 13 - Ensemble and final submission
# =====================================================================
val_probas = {
    "GBDT": proba_gbdt,
    "MLP": proba_mlp,
    "RNN": proba_rnn,
    "LSTM": proba_lstm,
    "Transformer": proba_trf,
    "MLP+GAN": proba_mlp_gan,
}
print("--- Individual models on the validation set ---")
for name, proba in val_probas.items():
    report(name, y_va, proba.argmax(1))

# --- Small grid search over blend weights ------------------------------
ordered = ["GBDT", "MLP", "RNN", "LSTM", "Transformer", "MLP+GAN"]
val_stack = np.stack([val_probas[k] for k in ordered])          # (M, N, 3)

best_w, best_score = None, -1.0
for w in np.ndindex(*([3] * len(ordered))):                      # weights in {0, 1, 2}
    if sum(w) == 0:
        continue
    blend = np.tensordot(np.array(w, dtype="float32"), val_stack, axes=1)
    score = balanced_accuracy(y_va, blend.argmax(1))
    if score > best_score:
        best_score, best_w = score, w

print("\n--- Ensemble ---")
print("best weights:", {k: int(v) for k, v in zip(ordered, best_w)})
print(f"ensemble balanced_acc = {best_score:.4f}")
report("ENSEMBLE", y_va, np.tensordot(np.array(best_w, dtype="float32"), val_stack, axes=1).argmax(1))

# --- Predict the test set with the same blend --------------------------
test_stack = np.stack([
    gbdt.predict_proba(X_te),          # GBDT
    predict_proba(mlp, X_te),          # MLP
    predict_proba(rnn, X_te),          # RNN
    predict_proba(lstm, X_te),         # LSTM
    predict_proba(transformer, X_te),  # Transformer
    predict_proba(mlp_gan, X_te),      # MLP + GAN
])
test_blend = np.tensordot(np.array(best_w, dtype="float32"), test_stack, axes=1)
pred_te = test_blend.argmax(1)

submission = pd.DataFrame({"id": test_df["id"],
                           "class": [INV_CLASS_MAP[i] for i in pred_te]})
submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv written:", submission.shape)
print(submission["class"].value_counts())
display(submission.head())


## 第 14 节 —— 收尾

### 你应该观察到的现象

| 模型 | 典型的验证集平衡准确率 | 备注 |
| :--- | ---: | :--- |
| KNN（k=100，第 3 节） | ≈ 0.93 | 排行榜上的起点 |
| 梯度提升树 | ≈ 0.950 | 最强的单个“经典”模型 |
| MLP | ≈ 0.940 | 需要 GPU 与若干轮训练 |
| RNN / LSTM | ≈ 0.930–0.945 | 把表格当序列读，略显刻意 |
| Transformer | ≈ 0.930–0.945 | 擅长序列，而非 14 个独立列 |
| MLP + GAN 增强 | ≈ 0.940–0.950 | 重平衡对稀有的 STAR 有帮助 |
| **加权集成** | **≈ 0.955+** | 通常是最好的那一个 |

*（具体数值会随随机种子和训练时长变化；各行的排序则非常稳定。）*

### 真正重要的两条教训

1. **让结构与数据相匹配。** 对表格数据而言，梯度提升树以远低于神经网络的代价，击败了我们试过的每一个神经网络。深度学习并非自动更好 —— 只有当数据具有与其归纳偏置相符的结构（序列、图像、语言）时，它才更好。
2. **优化你被打分的那个指标。** 从准确率切换到*平衡*准确率、给损失加权、用 GAN 做重平衡，都在把稀有的 STAR 类别往上推 —— 而这正是排行榜所衡量的东西。

### 值得尝试的扩展

* 对 `spectral_type` / `galaxy_population` 做**折外目标编码（out-of-fold target encoding）**（这个数据集上已知的排行榜技巧）。
* **堆叠（stacking）**：把所有基模型在验证集上的概率喂给一个小逻辑回归。
* **多种子平均**：每个神经网络用不同随机种子训练 3 次再平均。
* 用 **k 折交叉验证**替代单次划分，得到更可靠的分数估计。
* 把 `HistGradientBoostingClassifier` 换成 `LightGBM` / `XGBoost` / `CatBoost` —— 在表格数据上它们才是当前的最优选择。

### 这些思想今天都还在哪里

* MLP → 推荐系统、表格深度学习，以及每个 Transformer 的输出头。
* RNN / LSTM → 时间序列、传感器流、语音；现代序列模型的祖先。
* 注意力 / Transformer → GPT、BERT、ViT、蛋白质折叠 —— 这十年占据主导的结构。
* GAN → 图像合成、超分辨率、医学数据增强；也是扩散模型在思想上的先驱。

现在，你已经从一条单行决策规则，一路走到了一个训练好、集成好的深度学习流水线 —— 用的还是同一批来自 SDSS 望远镜的光子。
